# Tokenizers

Encode text to token ids, decode them, and turn a chat messages list into the string a model actually reads.

Llama 3.1 is gated. Accept the terms on [Meta-Llama-3.1-8B](https://huggingface.co/meta-llama/Meta-Llama-3.1-8B) and [Meta-Llama-3.1-8B-Instruct](https://huggingface.co/meta-llama/Meta-Llama-3.1-8B-Instruct). `HF_TOKEN` comes from the repo `.env`. These cells match `transformers==4.57.6`. No GPU is required.


## Setup

In [ ]:
import os
from pathlib import Path

from dotenv import load_dotenv

cwd = Path.cwd().resolve()
repo_root = next((p for p in [cwd, *cwd.parents] if (p / ".git").exists()), cwd)
load_dotenv(repo_root / ".env", override=True)

hf_home = repo_root / "W3-Open Source Models" / "hf-cache"
hf_home.mkdir(parents=True, exist_ok=True)
os.environ["HF_HOME"] = str(hf_home)
os.environ["HF_HUB_CACHE"] = str(hf_home / "hub")
os.environ["HF_DATASETS_CACHE"] = str(hf_home / "datasets")

from huggingface_hub import login
from transformers import AutoTokenizer

login(os.getenv("HF_TOKEN"))
print("Model cache:", hf_home)

LLAMA = "meta-llama/Meta-Llama-3.1-8B"
LLAMA_INSTRUCT = "meta-llama/Meta-Llama-3.1-8B-Instruct"
PHI4 = "microsoft/Phi-4-mini-instruct"
DEEPSEEK = "deepseek-ai/DeepSeek-V3.1"
QWEN_CODER = "Qwen/Qwen2.5-Coder-7B-Instruct"

messages = [
    {"role": "system", "content": "You are a helpful assistant"},
    {"role": "user", "content": "Tell a light-hearted joke for a room of Data Scientists"},
]


## Llama 3.1

One sentence, then the ids, then each id on its own.

In [ ]:
llama_base = AutoTokenizer.from_pretrained(LLAMA, trust_remote_code=True)

text = "I am excited to show Tokenizers in action to my LLM engineers"
tokens = llama_base.encode(text)
print(tokens)
print(
    f"There are {len(text)} characters, {len(text.split(' '))} words "
    f"and {len(tokens)} tokens"
)
print(llama_base.decode(tokens))
print(llama_base.batch_decode(tokens))
print(llama_base.get_added_vocab())
print(len(llama_base.vocab))


## Instruct chat template

Instruct models expect system, user, and assistant tags. `apply_chat_template` builds that string from the messages list. `add_generation_prompt=True` leaves the assistant turn open.

In [ ]:
llama = AutoTokenizer.from_pretrained(LLAMA_INSTRUCT, trust_remote_code=True)
prompt = llama.apply_chat_template(
    messages,
    tokenize=False,
    add_generation_prompt=True,
)
print(prompt)


## What the model actually receives

The messages list is not the model input. The chat template turns it into one string. The tokenizer turns that string into token ids. The model only sees those ids, and it predicts the next id.

## Llama, Phi, and DeepSeek

Same sentence, three tokenizers. Then the same messages list through each chat template.

In [ ]:
phi4 = AutoTokenizer.from_pretrained(PHI4)
deepseek = AutoTokenizer.from_pretrained(DEEPSEEK)

text = "I am curiously excited to show Hugging Face Tokenizers in action to my LLM engineers"


def show_pieces(name, tok, text):
    ids = tok.encode(text)
    print(name)
    print(ids)
    print(tok.batch_decode(ids))
    print()


def show_chat(name, tok, messages):
    prompt = tok.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=True,
    )
    print(name)
    print(prompt)
    print()


for name, tok in (("Llama", llama), ("Phi 4", phi4), ("DeepSeek", deepseek)):
    show_pieces(name, tok, text)

for name, tok in (("Llama", llama), ("Phi 4", phi4), ("DeepSeek", deepseek)):
    show_chat(name, tok, messages)


## Qwen coder

Each token in a small function, id then text.

In [ ]:
qwen = AutoTokenizer.from_pretrained(QWEN_CODER)
code = """
def hello_world(person):
  print("Hello", person)
"""
for token in qwen.encode(code):
    print(f"{token}={qwen.decode(token)}")
